# 23 · Fine-tuned transformer (E53)

Two ways to use a fine-tuned multilingual transformer (`multilingual-e5-small`, CPU, local weights):
1. **row as text**: platform, followers, likes, comments, shares and the post text in one string → log10(views) (the Kaggle "row as text" idea, as regression);
2. **text correction**: platform + text → residual of the final CatBoost model (inner out-of-fold residuals), added to CatBoost.

One joint model for the three platforms, 5 folds grouped by brand; CatBoost uses the same folds. Out-of-fold predictions are cached (`data/processed/finetune_oof_cache.pkl`, git-ignored).

In [1]:
import sys, json, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "reach_samples_students.csv").exists())
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.width", 220, "display.max_columns", 30)
import eda_utils as eu, extra_models as em, history as H, finetune_text as ft

In [2]:
f, meta = em.load(ROOT); raw = eu.load(ROOT).loc[f.index]
LOG = ROOT / "logs" / "finetune_progress.txt"
def log(s):
    print(s, flush=True); open(LOG, "a").write(s + "\n")
row, txt = ft.texts(f, meta, raw); print(len(row), "posts"); print(ft.CFG)

6735 posts
{'epochs': 3, 'batch': 32, 'max_len': 128, 'lr': 5e-05, 'lr_head': 0.001, 'seed': 0}


In [3]:
store = ft.run(f, meta, raw, ROOT / "data" / "processed" / "finetune_oof_cache.pkl", log)
pd.Series(store["seconds"]).round(0)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

epoch 1: train L1 0.436 (std units), 401 s


epoch 2: train L1 0.350 (std units), 817 s


epoch 3: train L1 0.324 (std units), 1242 s


row as text fold 0 done in 1271 s


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (41 outputs left out)

epoch 2: train L1 0.679 (std units), 789 s


epoch 3: train L1 0.618 (std units), 1183 s


text correction fold 4 done in 1207 s


row as text fold 0        1271.0
row as text fold 1        1384.0
row as text fold 2        1231.0
row as text fold 3        1207.0
row as text fold 4        1204.0
text correction fold 0    1205.0
text correction fold 1    1204.0
text correction fold 2    1205.0
text correction fold 3    1206.0
text correction fold 4    1207.0
dtype: float64

In [4]:
P = ft.table(store, meta); S = H.summarise(P)
S.pivot(index="model", columns="platform", values="MAE")[ft.PLATS].round(3)

platform,tiktok,youtube,twitter
model,,,
CatBoost (final model),0.271,0.453,0.467
CatBoost + half text correction,0.263,0.441,0.458
CatBoost + text correction,0.275,0.444,0.464
mean CatBoost + row as text,0.272,0.434,0.466
row as text,0.289,0.454,0.490


In [5]:
B = "CatBoost (final model)"
G = pd.DataFrame([{"platform": p, "model": k, **dict(zip(["gain", "CI low", "CI high"], H.gain_ci(P, p, B, k)))} for p in ft.PLATS for k in P.model.unique() if k != B]); G.round(3)

,platform,model,gain,CI low,CI high
0,tiktok,row as text,-0.018,-0.025,-0.012
1,tiktok,mean CatBoost + row as text,-0.001,-0.004,0.002
2,tiktok,CatBoost + text correction,-0.005,-0.011,0.002
3,tiktok,CatBoost + half text correction,0.007,0.004,0.011
4,youtube,row as text,-0.001,-0.015,0.013
5,youtube,mean CatBoost + row as text,0.019,0.012,0.027
6,youtube,CatBoost + text correction,0.009,0.001,0.017
7,youtube,CatBoost + half text correction,0.012,0.008,0.016
8,twitter,row as text,-0.023,-0.036,-0.010
9,twitter,mean CatBoost + row as text,0.001,-0.006,0.007


## Takeaways
See `docs/en/experiments.md` (E53) and `agent/vault/E53.md`. Numbers: `data/processed/finetune_text_numbers.json`.

In [6]:
json.dump({"config": ft.CFG, "models": S.to_dict("records"), "gains": G.to_dict("records"), "seconds": store["seconds"]}, open(ROOT / "data" / "processed" / "finetune_text_numbers.json", "w"), indent=1, default=float)